# 08_01 Inside an LSTM cell: what does one step actually compute?

An LSTM is four small layers and two lines of arithmetic. In this notebook you take a real `nn.LSTM` apart,
compute one step of it by hand from its own weights, and get exactly the numbers PyTorch gets. Then you
measure what the chapter claimed: how much of the error signal reaches the first word of a long sequence,
in a plain RNN, in an LSTM, and in an LSTM whose forget gate starts open.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-08-remembering-across-a-sentence", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'torch': 'torch',
           'sklearn': 'scikit-learn'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import torch
import torch.nn as nn
import headlines
from nlpcheck import ask, guess, reveal, check_08_01, lstm_step_inputs

torch.set_num_threads(4)
print("PyTorch", torch.__version__)

## 1. Recall

**r1.** Going back one step in time, a plain RNN multiplies the gradient by the recurrent weights and by what?
(a) the learning rate, (b) the slope of `tanh` at that step, which is at most 1, (c) the number of words

**r2.** What does gradient clipping do? (a) it drops the words whose gradient is too small, (b) it caps the
size of the whole gradient before each step, keeping its direction, (c) it stops training when the loss is
small

In [ ]:
ask("r1", "")
ask("r2", "")

## 2. The four blocks inside `nn.LSTM`

A tiny LSTM: 3 numbers per word in, a memory of 2. PyTorch keeps its weights in two matrices, `weight_ih_l0`
for the input word and `weight_hh_l0` for the last hidden state, each with the four gates stacked on top of
each other in the order **input, forget, candidate, output**. So each matrix has 4 x 2 = 8 rows.

In [ ]:
x, h0, c0, lstm = lstm_step_inputs()     # one word, the last hidden state and cell state, seeded weights
for name, p in lstm.named_parameters():
    print(f"{name:14} {tuple(p.shape)}")
W_ii, W_if, W_ig, W_io = lstm.weight_ih_l0.chunk(4)   # split the stack into its four gates
W_hi, W_hf, W_hg, W_ho = lstm.weight_hh_l0.chunk(4)
b = lstm.bias_ih_l0 + lstm.bias_hh_l0                 # the two bias vectors are always added
b_i, b_f, b_g, b_o = b.chunk(4)
print("x =", x.tolist(), " h0 =", h0.tolist(), " c0 =", c0.tolist())

## 3. One step by hand

The worked example: the input gate, the forget gate and the candidate, each one layer reading the word and the
last hidden state.

In [ ]:
with torch.no_grad():
    i = torch.sigmoid(x @ W_ii.T + h0 @ W_hi.T + b_i)   # how much of the candidate to write
    f = torch.sigmoid(x @ W_if.T + h0 @ W_hf.T + b_f)   # how much of the old memory to keep
    g = torch.tanh(x @ W_ig.T + h0 @ W_hg.T + b_g)      # what could be written
print("input gate i =", i.tolist())
print("forget gate f =", f.tolist())
print("candidate g =", g.tolist())

Your turn: the output gate `o` is built exactly like `i` and `f`, from `W_io`, `W_ho` and `b_o`. Then the two
lines that do all the remembering: `c1 = f * c0 + i * g`, and `h1 = o * tanh(c1)`.

In [ ]:
with torch.no_grad():
    o = None    # YOUR CODE HERE: the output gate, a sigmoid like i and f
    c1 = None   # YOUR CODE HERE: keep f of the old memory, add i of the candidate
    h1 = o * torch.tanh(c1) if o is not None and c1 is not None else None
    out, (h_pt, c_pt) = lstm(x.unsqueeze(0), (h0.unsqueeze(0), c0.unsqueeze(0)))
print("by hand:  h1 =", None if h1 is None else h1.tolist(), " c1 =", None if c1 is None else c1.tolist())
print("nn.LSTM:  h1 =", h_pt[0].tolist(), " c1 =", c_pt[0].tolist())

In [ ]:
results = {"h1": None if h1 is None else h1.tolist(), "c1": None if c1 is None else c1.tolist()}
os.makedirs("out", exist_ok=True)
json.dump(results, open("out/08_01_cell.json", "w"), indent=1)
check_08_01(part="cell")

Notice what did **not** happen to `c0` on its way to `c1`: no weight matrix, no `tanh`. It was multiplied by
`f` and had `i * g` added. That is the belt along the top of the chapter's picture, and it is the whole reason
for the next section.

## 4. How far back does the error reach?

`headlines.first_word_task` makes sequences of 40 made-up words whose label is decided by the **first** word
alone; everything after it is noise. To learn it, a network needs the first word's gradient. The cell below
measures the gradient's size at every position, for an untrained RNN, and prints the first word's as a
fraction of the last word's. Lab 07 measured about `1e-17` over 50 newswire words.

First, predict the same fraction for an untrained **LSTM**, as a power of ten, for example `1e-3`.

In [ ]:
guess("lstm_ratio", None)

In [ ]:
X, y = headlines.first_word_task(256, 40, seed=1)
def ratio(model):
    g = headlines.position_gradients(model, X, y)
    return float(g[0] / g[-1]), g
small = dict(vocab_size=headlines.TASK_VOCAB, embed_dim=16, hidden=32)

torch.manual_seed(0)
results["ratio_rnn"], g_rnn = ratio(headlines.SequenceClassifier(cell="rnn", **small))
torch.manual_seed(0)
results["ratio_lstm"], g_lstm = ratio(headlines.SequenceClassifier(cell="lstm", **small))
print(f"untrained RNN:  first word gets {results['ratio_rnn']:.0e} of the last word's gradient")
print(f"untrained LSTM: first word gets {results['ratio_lstm']:.0e} of the last word's gradient")
reveal("lstm_ratio", f"{results['ratio_lstm']:.0e}")

About `4e-11` for the RNN and `1e-08` for the LSTM: the LSTM keeps a few hundred times more of the gradient at
the first word, because the belt has no `tanh` on it. And it is still tiny, because a fresh LSTM's forget gates
sit near `sigmoid(0) = 0.5`, and forty halvings is about `1e-12`. (The measured figure is a little larger than
that because each gate also depends on the word and the state, so some slots sit above 0.5.)

## 5. Open the forget gate before training

`headlines.open_forget_gate(model, 3)` sets every forget gate's bias to 3, so every gate starts at
`sigmoid(3)`, about 0.95: remembering, rather than forgetting half. Predict the first word's fraction now.

In [ ]:
guess("open_ratio", None)

In [ ]:
torch.manual_seed(0)
opened = headlines.open_forget_gate(headlines.SequenceClassifier(cell="lstm", **small), 3.0)
results["ratio_lstm_open"], g_open = ratio(opened)
print(f"LSTM, forget gate open: first word gets {results['ratio_lstm_open']:.2f} of the last word's gradient")
reveal("open_ratio", f"{results['ratio_lstm_open']:.0e}")

About `0.54`: one number changed, and the first word went from a hundred-millionth of the last word's signal to
about half of it. This is
what "the LSTM solves the vanishing gradient" means, mechanically: it gives the gradient a path that is
multiplied only by the forget gate, and a forget gate near 1 passes it through. Whether the gate **stays**
near 1 is up to training, which is the next notebook.

In [ ]:
json.dump(results, open("out/08_01_cell.json", "w"), indent=1)
check_08_01()

## 6. Exit ticket

Explain it back, in the cell below, in two sentences of your own: why does the gradient survive along the cell
state when it vanishes along an RNN's hidden state?

**x1.** Which function decides what to throw away from the cell state? (a) tanh, (b) sigmoid, in the forget
gate, (c) ReLU

**x2.** What is the candidate `g`'s role? (a) it creates the new values that could be added to the cell state,
(b) it returns values between 0 and 1 that scale the memory, (c) it forgets the previous values

In [ ]:
my_explanation = ""
ask("x1", "")
ask("x2", "")